### Write a small program that implements a basic automatic differentiation engine from scratch, then use it to train an 8-bit autoencoder with an architecture of 8 to 3 to 8. Your inputs are 8 binary bits, 00000001, 00000010, 00000100, 00001000, 00010000, 00100000, 01000000, 10000000. Your expected outputs match the inputs. In other words, you're giving the network 00000010 and expecting it to produce 00000010, etc. You should use backpropagation for training. After successful training, your network should NOT make any mistakes (you give it a 00000010 and it always outputs 00000010, etc.). Now you can cut away the last layer, and you end up with a network that turns your input string into a 3-bit binary representation (Note that 00000010 will not correspond to 010 ('2' in binary)). Submit your code along with a log of output...


In [105]:
import math
import random

def makeValue(x):
    return {"value": float(x), "grad": 0.0, "parents": []}


def toValue(x):
    if(isinstance(x, dict)):
        return x
    return makeValue(x)


def add(a, b):
    a = toValue(a)
    b = toValue(b)
    result = makeValue(a["value"] + b["value"])
    result["parents"] = [(a, 1.0), (b, 1.0)]
    return result


def multiply(a, b):
    a = toValue(a)
    b = toValue(b)
    result = makeValue(a["value"] * b["value"])
    result["parents"] = [(a, b["value"]), (b, a["value"])]
    return result


def sigmoid(z):
    x = toValue(z)
    z = max(-500, min(500, x["value"]))
    value = 1 / (1 + math.exp(-z))
    derivative = value * (1 - value)
    result = makeValue(value)
    result["parents"] = [(x, derivative)]
    return result


def buildGraph(value, visited, graph):
    if id(value) in visited:
        return
    visited.add(id(value))
    for parent, _ in value["parents"]:
        buildGraph(parent, visited, graph)
    graph.append(value)


def backward(loss):
    visited = set()
    graph = []
    buildGraph(loss, visited, graph)
    loss["grad"] = 1.0
    for value in reversed(graph):
        for parent, derivative in value["parents"]:
            parent["grad"] += value["grad"] * derivative


def calculateLoss(output, expected):
    loss = makeValue(0)
    for i in range(len(output)):
        difference = add(output[i], multiply(expected[i], -1))
        squaredDifference = multiply(difference, difference)
        loss = add(loss, squaredDifference)
    return multiply(loss, 1 / len(output))


def forward(weight, inputData, bias):
    output = []
    zValue = []

    for i in range(len(weight)):
        value = makeValue(0)
        for j in range(len(inputData)):
            product = multiply(weight[i][j], inputData[j])
            value = add(value, product)
        z = add(value, bias[i])
        zValue.append(z)
        output.append(sigmoid(z))
    return output, zValue


def predict(hiddenWeight, hiddenBias, outputWeight, outputBias, inputData):
    inputNodes = [makeValue(x) for x in inputData]
    hidden, zHidden = forward(hiddenWeight, inputNodes, hiddenBias)
    output, zOutput = forward(outputWeight, hidden, outputBias)
    return hidden, output, zHidden, zOutput


def updateWeights(hiddenWeight, hiddenBias, outputWeight, outputBias, learningRate):
    for i in range(len(hiddenWeight)):
        for j in range(len(hiddenWeight[i])):
            hiddenWeight[i][j]["value"] -= learningRate * hiddenWeight[i][j]["grad"]
            hiddenWeight[i][j]["grad"] = 0.0
        hiddenBias[i]["value"] -= learningRate * hiddenBias[i]["grad"]
        hiddenBias[i]["grad"] = 0.0

    for i in range(len(outputWeight)):
        for j in range(len(outputWeight[i])):
            outputWeight[i][j]["value"] -= learningRate * outputWeight[i][j]["grad"]
            outputWeight[i][j]["grad"] = 0.0
        outputBias[i]["value"] -= learningRate * outputBias[i]["grad"]
        outputBias[i]["grad"] = 0.0


def hiddenLayer(hiddenWeight, hiddenBias, inputData):
    inputNodes = [makeValue(x) for x in inputData]
    hidden, _ = forward(hiddenWeight, inputNodes, hiddenBias)
    hLayer = [1 if value["value"] >= 0.5 else 0 for value in hidden]
    return hLayer


In [117]:
random.seed(42)
hiddenWeight = [[makeValue(random.uniform(-0.5, 0.5)) for _ in range(8)] for _ in range(3)]
hiddenBias = [makeValue(0.0) for _ in range(3)]

outputWeight = [[makeValue(random.uniform(-0.5, 0.5)) for _ in range(3)] for _ in range(8)]
outputBias = [makeValue(0.0) for _ in range(8)]

In [118]:

binary_8 = [[0,0,0,0,0,0,0,1],
            [0,0,0,0,0,0,1,0],
            [0,0,0,0,0,1,0,0],
            [0,0,0,0,1,0,0,0],
            [0,0,0,1,0,0,0,0],
            [0,0,1,0,0,0,0,0],
            [0,1,0,0,0,0,0,0],
            [1,0,0,0,0,0,0,0]]

learningRate = 1.0
trainingSteps = 10000

with open("output.txt", "w", encoding="utf-8") as f:
    print("Training Autoencoder")
    f.write("Training Autoencoder\n")
    for i in range(trainingSteps):
        totalLoss = 0
        for inputData in binary_8:
            expected = inputData
            _, output, _, _ = predict(hiddenWeight, hiddenBias, outputWeight, outputBias, inputData)
            loss = calculateLoss(output, expected)
            totalLoss += loss["value"]
            backward(loss)
            updateWeights(hiddenWeight, hiddenBias, outputWeight, outputBias, learningRate)

        if ((i + 1) % (trainingSteps//10) == 0):
            message = f"Training Step: {i + 1} | Loss: {totalLoss / len(binary_8)}"
            print(message)
            f.write(message + "\n")


    print("\nTesting Autoencoder (8->3->8)")
    f.write("\nTesting Autoencoder (8->3->8)\n")
    correct = 0
    for inputData in binary_8:
        _, output, _, _ = predict(hiddenWeight, hiddenBias, outputWeight, outputBias, inputData)
        reconstructed = [1 if value["value"] >= 0.5 else 0 for value in output]
        original = ''.join(str(bit) for bit in inputData)
        result = ''.join(str(bit) for bit in reconstructed)
        if(reconstructed == inputData):
            correct += 1
        message = f"Input: {original} | Output: {result}"
        print(message)
        f.write(message + "\n")
    message = f"Correct: {correct}/8"
    print(message)
    f.write(message + "\n")

    print("\nTesting Autoencoder (8->3)")
    f.write("\nTesting Autoencoder (8->3)\n")
    correct = 0
    encodedValues = []
    for inputData in binary_8:
        hLayer = hiddenLayer(hiddenWeight, hiddenBias, inputData)
        encodedValues.append(tuple(hLayer))
        original = ''.join(str(bit) for bit in inputData)
        result = ''.join(str(bit) for bit in hLayer)
        unique = len(set(encodedValues))
        message = f"Input: {original} | Output: {result}"
        print(message)
        f.write(message + "\n")

    message = f"Unique: {unique}/8"
    print(message)
    f.write(message + "\n")


Training Autoencoder
Training Step: 1000 | Loss: 0.03586513564201966
Training Step: 2000 | Loss: 0.025815700379825137
Training Step: 3000 | Loss: 0.00989597494458291
Training Step: 4000 | Loss: 0.0038440512770210617
Training Step: 5000 | Loss: 0.002376505050334779
Training Step: 6000 | Loss: 0.001699817963269585
Training Step: 7000 | Loss: 0.0013131151009632314
Training Step: 8000 | Loss: 0.001064844374194065
Training Step: 9000 | Loss: 0.0008929045400897017
Training Step: 10000 | Loss: 0.000767253189959226

Testing Autoencoder (8->3->8)
Input: 00000001 | Output: 00000001
Input: 00000010 | Output: 00000010
Input: 00000100 | Output: 00000100
Input: 00001000 | Output: 00001000
Input: 00010000 | Output: 00010000
Input: 00100000 | Output: 00100000
Input: 01000000 | Output: 01000000
Input: 10000000 | Output: 10000000
Correct: 8/8

Testing Autoencoder (8->3)
Input: 00000001 | Output: 000
Input: 00000010 | Output: 010
Input: 00000100 | Output: 110
Input: 00001000 | Output: 101
Input: 00010000